# Decoding sweep — beam `length_penalty` × `min_new_tokens`

Runs `04_decode.py --sweep` against a trained BanglaT5 checkpoint on the frozen
5,000-row dev split. Cheap points before touching MBR.

### Requires
| Input | Purpose |
|---|---|
| `nascenia-code` dataset | `metric.py`, `01_prep.py`, `04_decode.py` |
| Nascenia AI Hackathon competition | raw `train.csv` / `test.csv` for the dev split |
| **a checkpoint dataset** | trained weights — see below |

### ⚠️ The checkpoint
This notebook does **not** train. It needs an existing checkpoint attached as a Kaggle
Dataset or Model containing `config.json` + `model.safetensors`. Create one from a
finished training run's `/kaggle/working/runs/<name>/best` directory.

Cell 3 globs for it and fails loudly with instructions if it isn't attached — better
than burning a GPU session to discover it 40 minutes in.

### Judge on Token F1 / ROUGE-L, not composite
The local composite is mis-calibrated by ~0.118 (the organizers' BERTScore model is
unknown). Bars to beat:
- **Token F1 0.2669** — the constant string currently at #1 on the leaderboard
- **Token F1 0.3519** — a constant built from pure unigram frequencies (CONST-OPT-01).
  Below this, the model has learned nothing beyond word frequencies.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| n_gpu", torch.cuda.device_count())

In [ ]:
# csebuetnlp normalizer — inputs MUST be preprocessed exactly as in training,
# or the sweep measures the wrong thing.
!pip install -q git+https://github.com/csebuetnlp/normalizer
from normalizer import normalize
print("normalizer OK:", normalize("হেলো,   নাসেনিয়া ডকে  আপনাকে স্বাগতম।"))

In [ ]:
import glob, os, shutil, sys

print("/kaggle/input:", os.listdir("/kaggle/input"))

code_hits = glob.glob("/kaggle/input/**/04_decode.py", recursive=True)
assert code_hits, "attach the `nascenia-code` dataset (Add Input -> Datasets)"
CODE = os.path.dirname(code_hits[0])

raw_hits = glob.glob("/kaggle/input/**/train.csv", recursive=True)
assert raw_hits, "attach the Nascenia AI Hackathon competition (Add Input -> Competitions)"
RAW = os.path.dirname(raw_hits[0])

# find a checkpoint: any attached dir holding a config.json next to model weights,
# excluding the code dataset itself
cands = []
for cfg in glob.glob("/kaggle/input/**/config.json", recursive=True):
    d = os.path.dirname(cfg)
    if glob.glob(f"{d}/*.safetensors") or glob.glob(f"{d}/*.bin"):
        cands.append(d)

if not cands:
    raise FileNotFoundError(
        "No checkpoint found.\n"
        "This notebook does not train — attach a trained BanglaT5 checkpoint:\n"
        "  1. finish a training run\n"
        "  2. save /kaggle/working/runs/<name>/best as a Kaggle Dataset or Model\n"
        "  3. Add Input -> that dataset\n"
        f"searched /kaggle/input, found only: {os.listdir('/kaggle/input')}"
    )

CKPT = cands[0]
os.makedirs("/kaggle/working/code", exist_ok=True)
for f in glob.glob(f"{CODE}/*.py"):
    shutil.copy(f, "/kaggle/working/code/")
sys.path.insert(0, "/kaggle/working/code")

print("CODE:", CODE)
print("RAW :", RAW)
print("CKPT:", CKPT)
print("  contents:", sorted(os.listdir(CKPT))[:8])
if len(cands) > 1:
    print(f"  note: {len(cands)} checkpoints attached, using the first. all: {cands}")

In [ ]:
# Rebuild the frozen dev split with the same seed and code as training,
# so dev numbers stay comparable across machines.
!cd /kaggle/working/code && python 01_prep.py --raw "{RAW}" --out /kaggle/working/processed --seed 42 --dev-size 5000

## Quick sanity decode

200 rows, default settings. Confirms the checkpoint loads, the 3B assertion passes, and
output length is sane — before spending ~20 min on the full grid.

In [ ]:
!cd /kaggle/working/code && python 04_decode.py \
    --ckpt "{CKPT}" --split dev --mode beam \
    --limit 200 --no-bertscore

## Full sweep — 15 configs × 5,000 dev rows

`length_penalty` ∈ {0.6, 0.8, 1.0, 1.2, 1.5} × `min_new_tokens` ∈ {60, 80, 100}.

Measured length sensitivity says undershooting the ~93-token reference median costs
real points while overshooting costs almost nothing — so expect the optimum to sit at
the higher `min_new_tokens` values.

In [ ]:
!cd /kaggle/working/code && python 04_decode.py \
    --ckpt "{CKPT}" --split dev --sweep

## MBR at the best length setting

Fill in the winning `--length-penalty` / `--min-new-tokens` from the sweep above.
MBR selection costs ~8 min for 5,000 rows at 24 candidates (measured), on top of generation.

In [ ]:
BEST_LP, BEST_MNT = 1.0, 80   # <-- update from the sweep table

!cd /kaggle/working/code && python 04_decode.py \
    --ckpt "{CKPT}" --split dev --mode mbr -n 24 \
    --length-penalty {BEST_LP} --min-new-tokens {BEST_MNT} \
    --record /kaggle/working/mbr_dev.json

---
## After this

1. Record both results in `LOCAL_EXPERIMENTS.md` — all three metric components separately.
2. Compare **beam vs MBR** on Token F1 / ROUGE-L. MBR should win; if it doesn't, raise
   `-n`, or try `--utility f1`.
3. Only then generate a test submission with the winning configuration.